In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def collect_laacib_links(base_url, total_pages):
    all_links = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    for page_num in range(1, total_pages + 1):
        # Construct the paginated URL
        url = f"{base_url}page/{page_num}/"
        print(f"Fetching: {url}")
        
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Find all <a> tags with rel="bookmark"
                # This targets the specific post links you identified
                post_links = soup.find_all('a', rel='bookmark')
                
                page_links_count = 0
                for link in post_links:
                    href = link.get('href')
                    title = link.get('title') # Also grabbing the title for your NLP context
                    
                    if href and href not in [l['link'] for l in all_links]:
                        all_links.append({
                            'title': title,
                            'link': href,
                            'category': 'ciyaaro' # Tagging it for your dataset
                        })
                        page_links_count += 1
                
                print(f"Found {page_links_count} new links on page {page_num}.")
            else:
                print(f"Failed to load page {page_num}. Status: {response.status_code}")
                break # Stop if we hit a page that doesn't exist
                
        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            continue
            
        # Be polite to the server to avoid being blocked
        time.sleep(2)

    return all_links

# --- CONFIGURATION ---
# The base category URL (without the 'page/X/')
target_base_url = "https://www.laacibnet.net/category/wararka-premier-league/"
total_pages_to_scrape = 5  # Change this to however many pages you need

if __name__ == "__main__":
    links_data = collect_laacib_links(target_base_url, total_pages_to_scrape)
    
    # Save to Excel
    if links_data:
        df = pd.DataFrame(links_data)
        output_filename = "laacibnet_premier_league_links.xlsx"
        df.to_excel(output_filename, index=False)
        print(f"\nSuccess! Saved {len(links_data)} unique links to {output_filename}")
    else:
        print("No links collected.")

Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/1/
Found 17 new links on page 1.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/2/
Found 12 new links on page 2.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/3/
Found 12 new links on page 3.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/4/
Found 12 new links on page 4.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/5/
Found 12 new links on page 5.

Success! Saved 65 unique links to laacibnet_premier_league_links.xlsx


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

def collect_links_with_stagnation_check(categories_dict, output_file, max_stagnation=5):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Load existing data for Checkpoint
    if os.path.exists(output_file):
        df_master = pd.read_excel(output_file)
        all_links = df_master.to_dict('records')
        existing_urls = set(df_master['link'].tolist())
        print(f"Resuming: {len(existing_urls)} links already in database.")
    else:
        all_links = []
        existing_urls = set()

    for category_name, base_url in categories_dict.items():
        print(f"\n--- Processing Category: {category_name} ---")
        
        page = 1
        stagnation_counter = 0  # Tracks consecutive pages with 0 new links
        
        while True:
            url = f"{base_url}page/{page}/"
            print(f"Fetching Page {page}: {url}")
            
            try:
                response = requests.get(url, headers=headers, timeout=15)
                
                # 1. Stop if the server returns an error code (404, 500, etc.)
                if response.status_code != 200:
                    print(f"Reached end or error (Status: {response.status_code}).")
                    break
                
                soup = BeautifulSoup(response.content, 'html.parser')
                post_links = soup.find_all('a', rel='bookmark')
                
                # 2. Stop if the page is physically empty of links
                if not post_links:
                    print("Page is empty. Moving to next category.")
                    break

                new_on_page = 0
                for link in post_links:
                    href = link.get('href')
                    title = link.get('title')
                    
                    if href and href not in existing_urls:
                        all_links.append({
                            'title': title,
                            'link': href,
                            'category': category_name,
                            'source': 'laacibnet'
                        })
                        existing_urls.add(href)
                        new_on_page += 1
                
                # 3. Stagnation Logic
                if new_on_page == 0:
                    stagnation_counter += 1
                    print(f"Stagnation warning: {stagnation_counter}/{max_stagnation} pages with no new links.")
                else:
                    stagnation_counter = 0  # Reset counter if we find even one new link
                
                if stagnation_counter >= max_stagnation:
                    print(f"Stagnation limit reached for {category_name}. Skipping to next...")
                    break

                print(f"Added {new_on_page} new links from page {page}.")
                
                # Checkpoint Save
                pd.DataFrame(all_links).to_excel(output_file, index=False)
                
                page += 1
                time.sleep(1.5)
                
            except Exception as e:
                print(f"Error on {url}: {e}")
                break

    print(f"\nCompleted! Total unique links: {len(all_links)}")

# --- CONFIGURATION ---
categories = {
    "ciyaaraha_maanta": "https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/",
    "premier_league": "https://www.laacibnet.net/category/wararka-premier-league/",
    "la_liga": "https://www.laacibnet.net/category/wararka-la-liga/"
}

output_excel = "laacibnet_curated_dataset.xlsx"

if __name__ == "__main__":
    collect_links_with_stagnation_check(categories, output_excel)


--- Processing Category: ciyaaraha_maanta ---
Fetching Page 1: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/1/
Added 17 new links from page 1.
Fetching Page 2: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/2/
Added 12 new links from page 2.
Fetching Page 3: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/3/
Added 12 new links from page 3.
Fetching Page 4: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/4/
Added 12 new links from page 4.
Fetching Page 5: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/5/
Added 12 new links from page 5.
Fetching Page 6: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/6/
Added 12 new links from page 6.
Fetching Page 7: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/7/
Added 12 new links from page 7.
Fetching Page 8: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/8/
Added 12 new links from page 8.
Fetching Page 9: 

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

def collect_links_with_stagnation_check(categories_dict, output_file, max_stagnation=5):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Load existing data for Checkpoint
    if os.path.exists(output_file):
        df_master = pd.read_excel(output_file)
        all_links = df_master.to_dict('records')
        existing_urls = set(df_master['link'].tolist())
        print(f"Resuming: {len(existing_urls)} links already in database.")
    else:
        all_links = []
        existing_urls = set()

    for category_name, base_url in categories_dict.items():
        print(f"\n--- Processing Category: {category_name} ---")
        
        page = 1
        stagnation_counter = 0  # Tracks consecutive pages with 0 new links
        
        while True:
            url = f"{base_url}page/{page}/"
            print(f"Fetching Page {page}: {url}")
            
            try:
                response = requests.get(url, headers=headers, timeout=15)
                
                # 1. Stop if the server returns an error code (404, 500, etc.)
                if response.status_code != 200:
                    print(f"Reached end or error (Status: {response.status_code}).")
                    break
                
                soup = BeautifulSoup(response.content, 'html.parser')
                post_links = soup.find_all('a', rel='bookmark')
                
                # 2. Stop if the page is physically empty of links
                if not post_links:
                    print("Page is empty. Moving to next category.")
                    break

                new_on_page = 0
                for link in post_links:
                    href = link.get('href')
                    title = link.get('title')
                    
                    if href and href not in existing_urls:
                        all_links.append({
                            'title': title,
                            'link': href,
                            'category': category_name,
                            'source': 'laacibnet'
                        })
                        existing_urls.add(href)
                        new_on_page += 1
                
                # 3. Stagnation Logic
                if new_on_page == 0:
                    stagnation_counter += 1
                    print(f"Stagnation warning: {stagnation_counter}/{max_stagnation} pages with no new links.")
                else:
                    stagnation_counter = 0  # Reset counter if we find even one new link
                
                if stagnation_counter >= max_stagnation:
                    print(f"Stagnation limit reached for {category_name}. Skipping to next...")
                    break

                print(f"Added {new_on_page} new links from page {page}.")
                
                # Checkpoint Save
                pd.DataFrame(all_links).to_excel(output_file, index=False)
                
                page += 1
                time.sleep(1.5)
                
            except Exception as e:
                print(f"Error on {url}: {e}")
                break

    print(f"\nCompleted! Total unique links: {len(all_links)}")

# --- CONFIGURATION ---
categories = {
    "ciyaaraha_maanta": "https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/",
    "premier_league": "https://www.laacibnet.net/category/wararka-premier-league/",
    "la_liga": "https://www.laacibnet.net/category/wararka-la-liga/"
}

output_excel = "laacibnet_curated_dataset.xlsx"

if __name__ == "__main__":
    collect_links_with_stagnation_check(categories, output_excel)

Resuming: 622 links already in database.

--- Processing Category: ciyaaraha_maanta ---
Fetching Page 1: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/1/
Added 4 new links from page 1.
Fetching Page 2: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/2/
Stagnation warning: 1/5 pages with no new links.
Added 0 new links from page 2.
Fetching Page 3: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/3/
Stagnation warning: 2/5 pages with no new links.
Added 0 new links from page 3.
Fetching Page 4: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/4/
Stagnation warning: 3/5 pages with no new links.
Added 0 new links from page 4.


KeyboardInterrupt: 